# Tarea de Parquet

### 0. Inserción de dependencias

In [2]:
import pyarrow.parquet as pq
import pandas as pd

### 1. Convertir CSV a Parquet mediante Pandas

In [3]:
# Carga de CSV
df = pd.read_csv("in/2009.csv")
# Paso a Parquet
df.to_parquet("out/air2026EAT.parquet")

In [4]:
# Lectura del ParquetFile
pf = pq.ParquetFile("out/air2026EAT.parquet")
metadata = pf.metadata

cabecera = (f"  {'columna':<10} {'compr.':<8} {'codificación':<28} "
            f"{'codificado':>11} {'comprimido':>11} {'ratio':>6}  "
            f"{'min':<14} {'max':<14} {'nulos':>6} {'fisico':>6}")

for r in range(metadata.num_row_groups):
    rg = metadata.row_group(r)
    print(f"\n--- Row group {r}: {rg.num_rows} filas, {rg.total_byte_size/1024**2:.1f} MiB ---")
    print(cabecera)
    for i in range(rg.num_columns):
        col = rg.column(i)
        st = col.statistics
        sin, con = col.total_uncompressed_size, col.total_compressed_size
        linea = (f"  {col.path_in_schema:<10} {str(col.compression):<8} "
                 f"{', '.join(col.encodings):<28} "
                 f"{sin/1024:>7.1f} KiB {con/1024:>7.1f} KiB {sin/con:>5.1f}x  ")
        if st is None:
            print(linea + "(sin estadísticas)")
        else:
            print(linea + f"{str(st.min)[:14]:<14} {str(st.max)[:14]:<14} {st.null_count:>6} {st.physical_type:>6}")


--- Row group 0: 1048576 filas, 23.0 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  FL_DATE    SNAPPY   PLAIN, RLE, RLE_DICTIONARY       4.2 KiB     3.8 KiB   1.1x  2009-01-01     2009-03-02          0 BYTE_ARRAY
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       6.0 KiB     6.1 KiB   1.0x  9E             YV                  0 BYTE_ARRAY
  OP_CARRIER_FL_NUM SNAPPY   PLAIN, RLE, RLE_DICTIONARY    1724.8 KiB  1693.7 KiB   1.0x  1              7829                0  INT64
  ORIGIN     SNAPPY   PLAIN, RLE, RLE_DICTIONARY     845.9 KiB   844.0 KiB   1.0x  ABE            YUM                 0 BYTE_ARRAY
  DEST       SNAPPY   PLAIN, RLE, RLE_DICTIONARY    1127.8 KiB  1088.5 KiB   1.0x  ABE            YUM                 0 BYTE_ARRAY
  CRS_DEP_TIME SNAPPY   PLAIN, RLE, RLE_DICTIONARY    1422.8 KiB  1415.2 KiB   1.0x  3              2359                0  INT64
  DEP_TIME   SNAPPY   PLAIN, RLE, RLE_DI

#### Preguntas:
1. *Muestra la codificación (encodings) de cada columna. Verás que en todas aparece PLAIN. ¿Significa eso que ninguna usa diccionario? Explica a qué parte del column chunk corresponde cada uno de los nombres que aparecen.*

- No tiene por qué, debido a que en el encoding también aparece RLE_DICTIONARY, que usa diccionarios e índices para representar los valores.
- **Relación de los nombres a los column chunk**:
    - *PLAIN*: Almacena los valores directamente en las páginas de datos.
    - *RLE*: Codifica secuencias repetidas y metadatos internos de las páginas.
    - *RLE_DICTIONARY*: Representa valores mediante índices que apuntan a un diccionario creado con dichos valores.

2. *Compara el ratio de compresión (total_uncompressed_size / total_compressed_size) de una columna de baja cardinalidad (OP_CARRIER) y otra de alta (DEP_DELAY). ¿Cuál sale mejor parada? Justifica el resultado.*

- Ratio de compresión de OP_CARRIER: (6.0 / 6.1) = 0.98360x
- Ratio de compresión de DEP_DELAY: (1277.6 / 1267.6) = 1.007x
- Conclusión: DEP_DELAY sale mejor parada de la compresión, debido a que su ratio de compresión es más alto. También cabe recalcar que el ratio de compresión es menor a 1, por lo que OP_CARRIER no solo es que tenga peor compresión, sino que aparte su compresión le añade tamaño.

3. *Recupera y muestra el tipo físico (physical_type) y las estadísticas (min, max) de la columna FL_DATE. ¿Es el tipo que esperabas para una fecha? ¿Son correctas las estadísticas? Razona tu respuesta y corrige el problema reescribiendo el fichero como air\<anyo>\<iniciales>_fecha.parquet. Demuestra con los metadatos que ahora sí es correcto.*

- Tipo físico: BYTE-ARRAY
- Mínimo: 2009-01-01
- Máximo: 2009-12-31
- No es el tipo exacto que me esperaba para una fecha, debido a que es un formato de tipo String, pero las estadísticas (min-max) no las veo del todo mal, era lo que podía esperarme de un dataset de 2009, que el minimo fuera el 1 de enero de 2009 y el maximo el 31 de diciembre de 2009.

In [5]:
# Intento de arreglo de la estadística de la fecha:
import pyarrow.compute as pc
import pyarrow as pa
tabla = pq.read_table('out/air2026EAT.parquet')
fechas = pc.strptime(
    tabla["FL_DATE"],
    format="%Y-%m-%d",
    unit="s"
)

fechas = pc.cast(fechas, pa.date32())

idx = tabla.schema.get_field_index("FL_DATE")
tabla = tabla.set_column(idx, "FL_DATE", fechas)

pq.write_table(tabla, "out/air2026EAT_fecha.parquet")

# Lectura del ParquetFile
pf = pq.ParquetFile("out/air2026EAT_fecha.parquet")
metadata = pf.metadata

cabecera = (f"  {'columna':<10} {'compr.':<8} {'codificación':<28} "
            f"{'codificado':>11} {'comprimido':>11} {'ratio':>6}  "
            f"{'min':<14} {'max':<14} {'nulos':>6} {'fisico':>6}")

for r in range(metadata.num_row_groups):
    rg = metadata.row_group(r)
    print(f"\n--- Row group {r}: {rg.num_rows} filas, {rg.total_byte_size/1024**2:.1f} MiB ---")
    print(cabecera)
    for i in range(1):
        col = rg.column(i)
        st = col.statistics
        sin, con = col.total_uncompressed_size, col.total_compressed_size
        linea = (f"  {col.path_in_schema:<10} {str(col.compression):<8} "
                 f"{', '.join(col.encodings):<28} "
                 f"{sin/1024:>7.1f} KiB {con/1024:>7.1f} KiB {sin/con:>5.1f}x  ")
        if st is None:
            print(linea + "(sin estadísticas)")
        else:
            print(linea + f"{str(st.min)[:14]:<14} {str(st.max)[:14]:<14} {st.null_count:>6} {st.physical_type:>6}")


--- Row group 0: 1048576 filas, 23.0 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  FL_DATE    SNAPPY   PLAIN, RLE, RLE_DICTIONARY       3.6 KiB     3.8 KiB   1.0x  2009-01-01     2009-03-02          0  INT32

--- Row group 1: 1048576 filas, 22.9 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  FL_DATE    SNAPPY   PLAIN, RLE, RLE_DICTIONARY       3.6 KiB     3.7 KiB   1.0x  2009-03-02     2009-04-29          0  INT32

--- Row group 2: 1048576 filas, 23.1 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  FL_DATE    SNAPPY   PLAIN, RLE, RLE_DICTIONARY       3.6 KiB     3.7 KiB   1.0x  2009-04-29     2009-06-26          0  INT32

--- Row group 3: 1048576 filas, 23.2 MiB ---
  columna    compr.   codificación                  co

Ahora si, el tipo fisico de la tabla es un entero que representa la fecha

### 2: Filtrado Anticipado

1. *Sobre el air\<anyo>\<iniciales>.parquet de la actividad anterior, muestra el min y el max de la columna OP_CARRIER de todos los row groups. ¿Cuántos podría descartar una consulta que filtre por una aerolínea concreta? Explica por qué.*

In [6]:
# Lectura del ParquetFile
pf = pq.ParquetFile("out/air2026EAT.parquet")
metadata = pf.metadata

cabecera = (f"  {'columna':<10} {'compr.':<8} {'codificación':<28} "
            f"{'codificado':>11} {'comprimido':>11} {'ratio':>6}  "
            f"{'min':<14} {'max':<14} {'nulos':>6} {'fisico':>6}")

for r in range(metadata.num_row_groups):
    rg = metadata.row_group(r)
    print(f"\n--- Row group {r}: {rg.num_rows} filas, {rg.total_byte_size/1024**2:.1f} MiB ---")
    print(cabecera)
    for i in range(1, 2):
        col = rg.column(i)
        st = col.statistics
        sin, con = col.total_uncompressed_size, col.total_compressed_size
        linea = (f"  {col.path_in_schema:<10} {str(col.compression):<8} "
                 f"{', '.join(col.encodings):<28} "
                 f"{sin/1024:>7.1f} KiB {con/1024:>7.1f} KiB {sin/con:>5.1f}x  ")
        if st is None:
            print(linea + "(sin estadísticas)")
        else:
            print(linea + f"{str(st.min)[:14]:<14} {str(st.max)[:14]:<14} {st.null_count:>6} {st.physical_type:>6}")


--- Row group 0: 1048576 filas, 23.0 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       6.0 KiB     6.1 KiB   1.0x  9E             YV                  0 BYTE_ARRAY

--- Row group 1: 1048576 filas, 22.9 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       5.9 KiB     6.0 KiB   1.0x  9E             YV                  0 BYTE_ARRAY

--- Row group 2: 1048576 filas, 23.1 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       5.9 KiB     6.0 KiB   1.0x  9E             YV                  0 BYTE_ARRAY

--- Row group 3: 1048576 filas, 23.2 MiB ---
  columna    compr.   codificación        

*¿Cuántos podría descartar una consulta que filtre por una aerolínea concreta?*

2. Reescribe el fichero como air\<anyo>\<iniciales>_ordenado.parquet, esta vez ordenando previamente los datos por OP_CARRIER y fijando row_group_size=200_000.

In [7]:
table = pd.read_parquet("out/air2026EAT.parquet")

table = table.sort_values(["OP_CARRIER"])

table.to_parquet("out/air2026EAT_ordenado.parquet")

3. *Vuelve a mostrar las estadísticas por row group. Elige una aerolínea e indica, mirando únicamente los metadatos, qué row groups se leerán y cuáles no.*

In [9]:
# Comprobación:
# Lectura del ParquetFile
pf = pq.ParquetFile("out/air2026EAT_ordenado.parquet")
metadata = pf.metadata

cabecera = (f"  {'columna':<10} {'compr.':<8} {'codificación':<28} "
            f"{'codificado':>11} {'comprimido':>11} {'ratio':>6}  "
            f"{'min':<14} {'max':<14} {'nulos':>6} {'fisico':>6}")

for r in range(metadata.num_row_groups):
    rg = metadata.row_group(r)
    print(f"\n--- Row group {r}: {rg.num_rows} filas, {rg.total_byte_size/1024**2:.1f} MiB ---")
    print(cabecera)
    for i in range(1, 2):
        col = rg.column(i)
        st = col.statistics
        sin, con = col.total_uncompressed_size, col.total_compressed_size
        linea = (f"  {col.path_in_schema:<10} {str(col.compression):<8} "
                 f"{', '.join(col.encodings):<28} "
                 f"{sin/1024:>7.1f} KiB {con/1024:>7.1f} KiB {sin/con:>5.1f}x  ")
        if st is None:
            print(linea + "(sin estadísticas)")
        else:
            print(linea + f"{str(st.min)[:14]:<14} {str(st.max)[:14]:<14} {st.null_count:>6} {st.physical_type:>6}")


--- Row group 0: 1048576 filas, 30.7 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       2.4 KiB     2.5 KiB   1.0x  9E             B6                  0 BYTE_ARRAY

--- Row group 1: 1048576 filas, 30.6 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       2.4 KiB     2.5 KiB   1.0x  B6             EV                  0 BYTE_ARRAY

--- Row group 2: 1048576 filas, 30.4 MiB ---
  columna    compr.   codificación                  codificado  comprimido  ratio  min            max             nulos fisico
  OP_CARRIER SNAPPY   PLAIN, RLE, RLE_DICTIONARY       2.4 KiB     2.5 KiB   1.0x  EV             NW                  0 BYTE_ARRAY

--- Row group 3: 1048576 filas, 31.0 MiB ---
  columna    compr.   codificación        

Si eligiera, por ejemplo, la aerolinea "AA", solo se leería el RowGroup 0, debido a que el resto no contienen esa aerolínea (Del RowGroup 1 en adelante leen de la B6 a la YV)

4. *Tanto en el fichero original como el ordenado, mide con %%time tres lecturas y compáralas en una celda Markdown*

In [18]:
%%time
print("Lectura completa:")
tabla = pq.read_table('out/air2026EAT_ordenado.parquet')

Lectura completa:
CPU times: user 2.27 s, sys: 796 ms, total: 3.07 s
Wall time: 356 ms


In [19]:
%%time
print("Solo proyección")
tabla = pq.read_table('out/air2026EAT_ordenado.parquet', columns=["OP_CARRIER", "DEP_DELAY"])

Solo proyección
CPU times: user 151 ms, sys: 34.4 ms, total: 186 ms
Wall time: 76 ms


In [ ]:
%%time
print("Proyección y predicado") 
tabla = pq.read_table('out/air2026EAT_ordenado.parquet', columns=["OP_CARRIER", "DEP_DELAY"], filters=[('OP_CARRIER', '=', 'AA')])

Proyección y predicado
CPU times: user 67.4 ms, sys: 14.4 ms, total: 81.7 ms
Wall time: 54.1 ms


<table>
<legend>Tiempos de cada lectura</legend>
<tr><th>Tipo de lectura</th><th>Tiempo (ms)</th></tr>
<tr><td>Lectura completa</td><td>3070</td></tr>
<tr><td>Sólo proyección</td><td>188</td></tr>
<tr><td>Proyección + predicado</td><td>81.7</td></tr>
</table>